# RabTech Academy — Task 04
## Supervised Classification Modeling & Tuning

This notebook trains **four distinct classification architectures**, compares them with **Stratified K-Fold cross-validation**, tunes hyperparameters using **GridSearchCV**, evaluates the tuned champion with **Precision, Recall, F1, ROC-AUC, confusion matrix and ROC curve**, and serializes the final end-to-end model.

**Dataset:** UCI Adult / Census Income.


In [ ]:
# If needed:
# %pip install -q ucimlrepo pandas numpy scikit-learn matplotlib joblib

import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## 1. Load and prepare the UCI Adult dataset

In [ ]:
adult = fetch_ucirepo(id=2)
X = adult.data.features.copy()
y = adult.data.targets.copy()
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

# Normalize target labels such as '>50K.' to '>50K'.
y = y.astype(str).str.strip().str.rstrip(".")
y = (y == ">50K").astype(int)

print("X shape:", X.shape)
print("Class distribution:")
display(y.value_counts().rename(index={0: "<=50K", 1: ">50K"}))
display(X.head())


## 2. Hold out the test set before preprocessing

The test partition remains untouched during model comparison and hyperparameter tuning.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Numerical:", numeric_features)
print("Categorical:", categorical_features)


## 3. Leakage-safe preprocessing

In [ ]:
def make_preprocessor():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ])

def make_pipeline(model):
    return Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", model)
    ])


## 4. Train and compare four model architectures

All baseline scores below are obtained from the **training partition only** using the same Stratified 5-Fold splits. ROC-AUC is the primary validation metric; precision, recall and F1 are also reported.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1500, solver="liblinear", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, max_depth=18, min_samples_leaf=2,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.08, max_depth=3,
        random_state=RANDOM_STATE
    )
}

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

rows = []
for name, estimator in models.items():
    start = time.time()
    result = cross_validate(
        make_pipeline(estimator),
        X_train, y_train,
        cv=CV, scoring=scoring,
        n_jobs=1, return_train_score=False
    )
    rows.append({
        "Model": name,
        "Precision": result["test_precision"].mean(),
        "Recall": result["test_recall"].mean(),
        "F1": result["test_f1"].mean(),
        "ROC_AUC": result["test_roc_auc"].mean(),
        "ROC_AUC_STD": result["test_roc_auc"].std(),
        "Seconds": time.time() - start
    })

comparison = (
    pd.DataFrame(rows)
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)
display(comparison)


## 5. Select model family from validation results

The champion **family** is selected from the highest mean cross-validated ROC-AUC. This avoids selecting a model based on the held-out test set.


In [ ]:
best_family = comparison.iloc[0]["Model"]
print("Best baseline family by CV ROC-AUC:", best_family)


## 6. Hyperparameter optimization with GridSearchCV

We tune the selected family using the same stratified 5-fold strategy and ROC-AUC scoring. Search spaces are intentionally compact so the notebook remains practical on a student computer.


In [ ]:
param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1.0, 10.0],
        "model__class_weight": [None, "balanced"]
    },
    "Decision Tree": {
        "model__max_depth": [6, 12, 20, None],
        "model__min_samples_leaf": [1, 5, 10],
        "model__class_weight": [None, "balanced"]
    },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [12, 20, None],
        "model__min_samples_leaf": [1, 3],
        "model__class_weight": [None, "balanced"]
    },
    "Gradient Boosting": {
        "model__n_estimators": [80, 120],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3]
    }
}

search = GridSearchCV(
    estimator=make_pipeline(models[best_family]),
    param_grid=param_grids[best_family],
    scoring="roc_auc",
    cv=CV,
    n_jobs=-1,
    refit=True,
    return_train_score=False
)

search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_, 4))

champion_model = search.best_estimator_


## 7. Final evaluation on the untouched test set

In [ ]:
y_pred = champion_model.predict(X_test)
y_prob = champion_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "ROC-AUC"],
    "Score": [
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ]
})
display(test_metrics)
print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))


### Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["<=50K", ">50K"]
)
plt.title("Champion Model — Confusion Matrix")
plt.show()


### ROC-AUC Curve

In [ ]:
RocCurveDisplay.from_predictions(
    y_test, y_prob,
    name=f"Tuned {best_family}"
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.title("Champion Model — ROC Curve")
plt.legend()
plt.show()


## 8. ROC curves for all four baseline architectures

For visual comparison, each baseline architecture is fitted on the training partition and evaluated on the same untouched test set. **These curves are descriptive only**; model selection above was based on training-set cross-validation, not these test curves.


In [ ]:
plt.figure(figsize=(8, 6))

for name, estimator in models.items():
    fitted = make_pipeline(clone(estimator))
    fitted.fit(X_train, y_train)
    prob = fitted.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=plt.gca())

plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.title("ROC Curves — Four Baseline Architectures")
plt.legend()
plt.show()


## 9. Serialize the champion model

The complete fitted pipeline—including imputation, scaling, one-hot encoding and classifier—is saved, so future raw records can be passed directly to the artifact.


In [ ]:
MODEL_PATH = "champion_model.joblib"
joblib.dump(champion_model, MODEL_PATH)
print("Saved:", MODEL_PATH)

# Verification
loaded_model = joblib.load(MODEL_PATH)
sample_predictions = loaded_model.predict(X_test.head(5))
print("Reloaded-model predictions:", sample_predictions)


## 10. Conclusion

This experiment compared four supervised classification architectures under the same stratified cross-validation protocol. The champion family was selected by mean validation ROC-AUC, tuned using `GridSearchCV`, and evaluated once on the held-out test set.

### Leakage controls
- Train/test split occurs before preprocessing.
- Imputers, scaler and encoder are contained inside each CV pipeline.
- Hyperparameters are selected using training folds only.
- The test set is not used for champion selection.
- The serialized artifact contains the complete preprocessing + model pipeline.

### Responsible-use note
The Adult dataset contains demographic attributes including race and sex. This notebook is an educational modeling exercise. High predictive performance does not establish fairness, legality, or suitability for consequential real-world decisions.
